# SCATS Volume Data — 5-min to 15-min aggregation


In [12]:
INPUT_FILE = r"C:\Users\IvanVelilla\Mobility Lab Limited\Projects - 1034 - Western LX Aimsun\New North Road\Received\SCATS NNR\SCATS_REPORT_2025Sept.xls"
OUTPUT_FILE = r"C:\Users\IvanVelilla\Documents\Projects\Western LX\outputs\scats_counts_15min_2050.xlsx"

# Set to a date string "YYYY-MM-DD" to filter to a single day, or None for all dates.
# Ignored when AVERAGE_DATES is set.
FILTER_DATE = ["2025-09-15"]

# Set to a list of date strings to average counts across those specific dates.
# e.g. ["2025-09-08", "2025-09-09", "2025-09-10"]
# When set, FILTER_DATE is ignored and the output contains averaged values per time slot.
AVERAGE_DATES = None

# Time window for the Summary sheet — "HH:MM" (24-hour). End is exclusive.
# Set both to None to summarise the full exported period.
SUMMARY_START_TIME = None
SUMMARY_END_TIME   = None

In [13]:
import pandas as pd


## Load and parse sheets


In [14]:
def load_sheet(path, sheet_name):
    """Read one SCATS volume sheet and return a wide 5-min dataframe.

    Columns returned: datetime, site, 1, 2, ... N  (one column per detector)
    """
    raw = pd.read_excel(path, sheet_name=sheet_name, header=None)

    site_id = int(raw.iloc[1, 1])     # row 1, col B = site number
    header_row = raw.iloc[2]          # row 2 has column labels
    detector_cols = [int(c) for c in header_row[3:].values]

    data = raw.iloc[3:].copy()
    data = data[data.iloc[:, 0].notna()]   # drop any trailing blank rows

    # Col 0 is already a datetime (Excel parsed it); col 1 is the "HH:MM - HH:MM" label
    data = data.rename(columns={0: "datetime"})
    data["datetime"] = pd.to_datetime(data["datetime"])

    # Keep only the detector count columns (skip time-label col 1 and Detector# col 2)
    det_data = data.iloc[:, 3:].copy()
    det_data.columns = detector_cols
    det_data = det_data.apply(pd.to_numeric, errors="coerce")

    det_data.insert(0, "datetime", data["datetime"].values)
    det_data.insert(1, "site", site_id)
    return det_data.reset_index(drop=True)


xl = pd.ExcelFile(INPUT_FILE)
print("Sheets found:", xl.sheet_names)
frames = [load_sheet(INPUT_FILE, s) for s in xl.sheet_names]
df5 = pd.concat(frames, ignore_index=True)
print(f"Loaded {len(df5):,} rows | {df5['datetime'].min()} → {df5['datetime'].max()}")
df5.head(10)


Sheets found: ['Site 2409', 'Site 2050']
Loaded 17,252 rows | 2025-09-01 00:00:00 → 2025-09-30 23:55:00


,datetime,site,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
0,2025-09-01 00:00:00,2409,4.0,4.0,0.0,2.0,2.0,3.0,0.0,1.0,4.0,2.0,2.0,0.0,0.0,0.0,0.0,0.0
1,2025-09-01 00:05:00,2409,3.0,4.0,0.0,2.0,1.0,1.0,0.0,0.0,2.0,3.0,1.0,0.0,0.0,0.0,0.0,0.0
2,2025-09-01 00:10:00,2409,6.0,6.0,1.0,4.0,0.0,1.0,0.0,0.0,7.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2025-09-01 00:15:00,2409,3.0,5.0,0.0,1.0,0.0,2.0,0.0,0.0,3.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2025-09-01 00:20:00,2409,1.0,4.0,0.0,1.0,0.0,2.0,0.0,0.0,1.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0
5,2025-09-01 00:25:00,2409,2.0,2.0,0.0,2.0,1.0,3.0,0.0,0.0,2.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0
6,2025-09-01 00:30:00,2409,0.0,3.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,2.0,3.0,0.0,0.0,0.0,0.0,0.0
7,2025-09-01 00:35:00,2409,2.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,2025-09-01 00:40:00,2409,3.0,1.0,0.0,2.0,1.0,5.0,0.0,0.0,3.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0
9,2025-09-01 00:45:00,2409,2.0,4.0,1.0,2.0,0.0,0.0,1.0,1.0,2.0,3.0,0.0,1.0,0.0,0.0,0.0,0.0


## Aggregate to 15-minute intervals


In [15]:
# Sum three consecutive 5-min bins into each 15-min interval.
# The interval label is the start of each 15-min window (floor to 15T).
df5["interval_15min"] = df5["datetime"].dt.floor("15min")

detector_cols = [c for c in df5.columns if isinstance(c, int)]

df15 = (
    df5.groupby(["interval_15min", "site"], sort=True)[detector_cols]
    .sum()
    .reset_index()
)

print(f"15-min rows: {len(df15):,}")
print(f"Sites: {sorted(df15['site'].unique())}")
print(f"Date range: {df15['interval_15min'].min()} → {df15['interval_15min'].max()}")
df15.head(10)


15-min rows: 5,752
Sites: [np.int64(2050), np.int64(2409)]
Date range: 2025-09-01 00:00:00 → 2025-09-30 23:45:00


,interval_15min,site,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
0,2025-09-01 00:00:00,2050,1.0,7.0,3.0,4.0,0.0,10.0,4.0,3.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2025-09-01 00:00:00,2409,13.0,14.0,1.0,8.0,3.0,5.0,0.0,1.0,13.0,7.0,3.0,0.0,0.0,0.0,0.0,0.0
2,2025-09-01 00:15:00,2050,1.0,2.0,2.0,2.0,0.0,9.0,3.0,8.0,13.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2025-09-01 00:15:00,2409,6.0,11.0,0.0,4.0,1.0,7.0,0.0,0.0,6.0,7.0,1.0,1.0,0.0,0.0,0.0,0.0
4,2025-09-01 00:30:00,2050,0.0,6.0,1.0,4.0,0.0,3.0,0.0,2.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,2025-09-01 00:30:00,2409,5.0,4.0,0.0,3.0,4.0,6.0,0.0,0.0,5.0,2.0,5.0,0.0,0.0,0.0,0.0,0.0
6,2025-09-01 00:45:00,2050,1.0,5.0,0.0,3.0,0.0,1.0,1.0,2.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,2025-09-01 00:45:00,2409,5.0,6.0,2.0,5.0,4.0,4.0,1.0,1.0,5.0,6.0,4.0,1.0,0.0,0.0,0.0,0.0
8,2025-09-01 01:00:00,2050,0.0,1.0,3.0,5.0,1.0,2.0,1.0,2.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,2025-09-01 01:00:00,2409,1.0,6.0,0.0,1.0,2.0,5.0,0.0,0.0,1.0,3.0,2.0,0.0,0.0,0.0,0.0,0.0


## Save output


In [16]:
export = df15.copy()
avg_mode = False

# Unwrap FILTER_DATE if accidentally passed as a list (e.g. [['2025-09-10']] or ['2025-09-10'])
_fd = FILTER_DATE
while isinstance(_fd, list):
    _fd = _fd[0] if _fd else None
FILTER_DATE = _fd

if AVERAGE_DATES is not None:
    target_dates = [pd.Timestamp(d).date() for d in AVERAGE_DATES]
    filtered = export[export["interval_15min"].dt.date.isin(target_dates)].copy()
    if filtered.empty:
        raise ValueError(
            f"No data found for {AVERAGE_DATES}. "
            f"Available range: {df15['interval_15min'].min().date()} → {df15['interval_15min'].max().date()}"
        )
    missing = [str(d) for d in target_dates if d not in filtered["interval_15min"].dt.date.unique()]
    if missing:
        print(f"Warning: no data for {missing}")
    filtered["time_of_day"] = filtered["interval_15min"].dt.time
    det_cols = [c for c in filtered.columns if isinstance(c, int)]
    export = (
        filtered.groupby(["time_of_day", "site"])[det_cols]
        .mean()
        .round(1)
        .reset_index()
    )
    avg_mode = True
    print(f"Averaged {len(AVERAGE_DATES)} date(s): {', '.join(AVERAGE_DATES)} — {len(export):,} rows")

elif FILTER_DATE is not None:
    target = pd.Timestamp(FILTER_DATE).date()
    export = export[export["interval_15min"].dt.date == target]
    if export.empty:
        raise ValueError(
            f"No data found for {FILTER_DATE}. "
            f"Available range: {df15['interval_15min'].min().date()} → {df15['interval_15min'].max().date()}"
        )
    print(f"Filtered to {FILTER_DATE} — {len(export):,} rows")

else:
    print(f"No date filter — exporting all {len(export):,} rows")

with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:

    # --- Per-site sheets ---
    for site, grp in export.groupby("site"):
        sheet = grp.drop(columns="site").reset_index(drop=True)
        det_cols = [c for c in sheet.columns if isinstance(c, int)]
        active = [c for c in det_cols if sheet[c].sum() > 0]

        if avg_mode:
            sheet = sheet[["time_of_day"] + active].rename(columns={"time_of_day": "time"})
        else:
            sheet = sheet[["interval_15min"] + active]
            sheet.insert(0, "date", sheet["interval_15min"].dt.date)
            sheet.insert(1, "time", sheet["interval_15min"].dt.time)
            sheet = sheet.drop(columns="interval_15min")

        sheet.to_excel(writer, sheet_name=f"Site {site}", index=False)
        print(f"Site {site}: {len(sheet):,} rows, detectors {active}")

    # --- Summary sheet ---
    det_cols_all = [c for c in export.columns if isinstance(c, int)]

    # Get the time series for filtering
    if avg_mode:
        time_series = export["time_of_day"]
    else:
        time_series = export["interval_15min"].dt.time

    if SUMMARY_START_TIME is not None and SUMMARY_END_TIME is not None:
        start_t = pd.Timestamp(SUMMARY_START_TIME).time()
        end_t   = pd.Timestamp(SUMMARY_END_TIME).time()
        mask = (time_series >= start_t) & (time_series < end_t)
        summary_src = export[mask].copy()
        period_label = f"{SUMMARY_START_TIME}–{SUMMARY_END_TIME}"
    else:
        summary_src = export.copy()
        period_label = "full period"

    summary = (
        summary_src.groupby("site")[det_cols_all]
        .sum()
        .round(1)
        .reset_index()
    )
    # Drop detectors with zero activity across all sites
    active_dets = [c for c in det_cols_all if summary[c].sum() > 0]
    summary = summary[["site"] + active_dets]

    # Write with period label in cell A1, table starting at row 2
    summary.to_excel(writer, sheet_name="Summary", index=False, startrow=1)
    ws = writer.sheets["Summary"]
    ws.cell(row=1, column=1, value=f"Period: {period_label}")
    print(f"Summary sheet: {len(summary)} sites, period {period_label}, detectors {active_dets}")

print(f"\nSaved → {OUTPUT_FILE}")

Filtered to 2025-09-15 — 192 rows
Site 2050: 96 rows, detectors [1, 2, 3, 4, 5, 6, 7, 8, 9]
Site 2409: 96 rows, detectors [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
Summary sheet: 2 sites, period full period, detectors [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]

Saved → C:\Users\IvanVelilla\Documents\Projects\Western LX\outputs\scats_counts_15min_2050.xlsx
